In [9]:
import pandas as pd 
from sqlalchemy import create_engine,MetaData,Column,Table,Integer,String,Float,text
from sqlalchemy.orm import sessionmaker,Session
import urllib

In [5]:
def get_engine():
    params=urllib.parse.quote_plus(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=localhost\\SQLEXPRESS;'
    'DATABASE=UZAIR;'
    'Trusted_Connection=yes;'
    'TrustServerCertificate=yes;')
    engine=create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
    return engine

In [7]:
engine=get_engine()

In [15]:
session=sessionmaker(bind=engine)

In [17]:
meta=MetaData()
bronze_current_weather=Table(
    'current_weather',
    meta,
    Column('recorded_at',String(50)),
    Column('temperature',String(50)),
    Column('windspeed',String(50)),
    Column('winddirection',String(100)),
    Column('is_day',String(50)),
    Column('weather_code',String(50)),
    Column('city',String(50)),
    Column('latitude',String(100)),
    Column('load_timestamp',String(100)),
    Column('weather_discription',String(100)),
)
bronze_hourly_weather=Table(
    'hourly_weather',
    meta,
    Column('forcast_time',String(50)),
    Column('temperature',String(50)),
    Column('percipitation',String(50)),
    Column('windspeed',String(50)),
    Column('weathercode',String(50)),
    Column('city',String(50)),
    Column('load_timestamp',String(50)),
    Column('weather_discription',String(100))

)
def init_database():
    meta.create_all(engine)
    print("database is ready")
init_database()    

database is ready


In [18]:
def load_file(file_path, target_table, engine, chunk_size=50000):
    df=pd.read_csv(file_path)
    records=df.to_dict(orient='records')
    with session(engine) as session:
        session.execute(target_table.delete())
        for i in range(0, len(records), chunk_size):
            chunk = records[i:i + chunk_size]
            session.execute(target_table.insert(), chunk)
            print(f"Inserted rows {i} to {i + len(chunk)}")
        session.commit()    
        

In [20]:
import pandas as pd
df = pd.read_csv(r"F:\pakistan_weather_pipeline\data\clean\current_weather.csv")
print(df.columns.tolist())

['recorded_at', 'temperature', 'windspeed', 'winddirection', 'is_day', 'weathercode', 'city', 'latitude', 'load_timestamp', 'weather_description']


In [19]:
def run_load():
    engine = get_engine()
    init_database()
    
    load_file(
        file_path=r"F:\pakistan_weather_pipeline\data\clean\current_weather.csv",
        target_table=bronze_current_weather,
        engine=engine
    )
    
    load_file(
        file_path=r"F:\pakistan_weather_pipeline\data\clean\hourly_weather.csv",
        target_table=bronze_hourly_weather,
        engine=engine
    )